# Comparing Inference and Training FLOPs

In this post, I show code to evaluate pretraining vs inference costs of GPT-4, and what user base size implications there are for relative cost.

Note, my inference calculations make only the assumption that we are doing KV caching as an optimization. We also assume that the underlying GPT-4 model has not changed over time to make the comparison fair.

## 1. Training Forward Pass

When training a dense LLM, we have a very simple setup - we fix B, the batch_size, and T, the maximum context length of the model. We then feed a (B,T) shape tensor of token indices into the model for the forward pass.

Appendix F of the [Chinchilla paper](https://arxiv.org/pdf/2203.15556.pdf) contains a detailed breakdown of the FLOPs in this forward.

To calculate the backwards pass FLOPs for training, we will use the standard assumption that it is 2x the forward FLOPs. This assumption was used in Chinchilla, and was originally asserted without justification in [Scaling Laws for Neural Language Models](https://arxiv.org/pdf/2001.08361.pdf) from OpenAI.

To give a quick theoretical justification of this as neither paper does so (it is well grounded in practical measurements):

Have $y=xW$ as forward, so one matmul. But when doing backprop we need to find 2 different quantities: $\dfrac{\delta L}{\delta W}$ (to learn weights) and $\dfrac{\delta L}{\delta x}$ (to continue to next layer down). Each of these two calculations involves one incremental matmul, for a total of 2. (Note bias is immaterial to flop counts so I ignore it)


## 2. Inference Forward Pass

The inference forward pass with KV caching is a little different. Given M input tokens and N output tokens, the first output token incurs a full forward transformer call over M parallel tokens. However, the subsequent N-1 tokens skip the full attention cost due to KV caching, and only incur compute costs for a single token.

## 3. GPT-4 training assumptions

We assume that GPT-4 was trained on 8192 context length (32K came from RoPE style interpolation on this I would guess).

Further, we assume that GPT-4 is 2 out of 16 MoE, where each expert sees 111bln params across all layers, and that it was trained for 13T tokens.

This seems a pretty well-validate rumour.

From this 111bln expert estimate I've backed out the likely feed-forward hyperparams.



In [30]:
from decimal import Decimal
## Code evaluating flops of an LLM ##


def forward_single_layer_flops(seq_len,d_model,ffd_ratio,moe=2):
    total=0
    # QKV projections in attention
    total+=2*3*seq_len*d_model**2

    # Key @ Query
    total+=2*seq_len**2*d_model

    # Softmax Outputs @ Values
    total+=2*seq_len**2*d_model
       
    # Final Linear in Attention
    total+=2*seq_len*d_model**2
    
    # Dense block
    total+=moe * 4 * seq_len * d_model**2*ffd_ratio

    return total

def forward_transformer_flops(n_layer,seq_len,d_model,ffd_ratio,vocab_size,kv_cached=False,moe=2):
    total =0

    # Embedding
    total+=2*seq_len*vocab_size*d_model
    
    # Main transformer Blocks
    total+=n_layer*forward_single_layer_flops(seq_len,d_model,ffd_ratio,moe)

    # Logits
    total+=2*seq_len*d_model*vocab_size

    if kv_cached:
        total/=seq_len

    return total

def single_sequence_transformer_flops(n_layer,seq_len,d_model,ffd_ratio,vocab_size,kv_cached=False,training=True,moe=2):
    factor=3 if training else 1
    return factor * forward_transformer_flops(n_layer,seq_len,d_model,ffd_ratio,vocab_size,kv_cached,moe)


# Common params

# estimate based on 111B expert, cf Gopher model at 280bln Params across MLP, Embeddings, Attention
d_model=12288
n_layer=92
ffd_ratio=4
moe=2 # each token sees this many experts

# other common params
vocab_size=10**5

# training params based on rumours
seq_len=8192
token_count=13*10**12 

total_batches=token_count/seq_len

# inference params from Coatue hypothetical
mean_input_len=150
mean_output_len=150 # Coatue only gave input length assumption, I assume output length same 
daily_user_count=100*10**6 
daily_queries_per_user=10


total_training_flops=total_batches*single_sequence_transformer_flops(n_layer,seq_len,d_model,ffd_ratio,vocab_size,False,True,moe)

daily_inference_flops=daily_user_count*daily_queries_per_user*single_sequence_transformer_flops(n_layer,mean_input_len,d_model,ffd_ratio,vocab_size,False,False,moe) # first token, no KV caching
daily_inference_flops+=daily_user_count*daily_queries_per_user*(mean_output_len-1)*single_sequence_transformer_flops(n_layer,mean_input_len,d_model,ffd_ratio,vocab_size,True,False,moe) # follow up tokens, KV caching

print("My Total Training FLOPs:",'%.2E' % Decimal(total_training_flops))
print("Coatue Training FLOPs:",'%.2E' % Decimal(2.1*10**25))
print("--------------------------")
print("My Daily Inference FLOPs:",'%.2E' % Decimal(daily_inference_flops))
print("Coatue Daily Inference FLOPs:",'%.2E' % Decimal(8*10**25))

My Total Training FLOPs: 2.33E+25
Coatue Training FLOPs: 2.10E+25
--------------------------
My Daily Inference FLOPs: 1.68E+23
Coatue Daily Inference FLOPs: 8.00E+25
